# SHAP Comparisons

In [1]:
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Models And Scaler Loadings

In [2]:
models = {
    "Logistic Regression": joblib.load("models/logistic_regression_model.pkl"),
    "KNN": joblib.load("models/KNN_model.pkl"),
    "MLP": joblib.load("models/MLP_model.pkl"),
    "Random Forest": joblib.load("models/random_forest_model.pkl"),
    "XGBoost": joblib.load("models/XGBoost_model.pkl"),
}

In [3]:
scalers = {
    "Logistic Regression": joblib.load("models/lrscaler.pkl"),
    "KNN": joblib.load("models/KNNscaler.pkl"),
    "MLP": joblib.load("models/mlpscaler.pkl"),
    "Random Forest": None,
    "XGBoost": None
}

In [4]:
X_train=pd.read_csv("X_train.csv")
X_test=pd.read_csv("X_test.csv")

In [8]:
X_eval = X_test.sample(100, random_state=42)
X_bg = X_train.sample(100, random_state=42)

In [6]:
def scale(name, X):
    sc = scalers[name]
    if sc is None:
        return X
    return pd.DataFrame(sc.transform(X), columns=X.columns, index=X.index)

## Computing SHAP

In [10]:
shap_values = {}

for name, model in models.items():
    print("working on", name)
    Xe = scale(name, X_eval)
    Xb = scale(name, X_bg)

    try:
        sv = shap.TreeExplainer(model)(Xe, check_additivity=False)
    except Exception:
        if isinstance(model, tf.keras.Model):
            predict = lambda d: model.predict(
                pd.DataFrame(d, columns=Xe.columns), verbose=0
            ).reshape(-1)
        else:
            predict = lambda d: model.predict_proba(
                pd.DataFrame(d, columns=Xe.columns)
            )[:, 1]

        sv = shap.Explainer(predict, Xb)(Xe)

    if sv.values.ndim == 3:
        sv = sv[:, :, 1]

    sv.data = X_eval.values
    shap_values[name] = sv

print("done")

working on Logistic Regression
working on KNN


PermutationExplainer explainer: 101it [1:46:42, 64.03s/it]                         


working on MLP


AttributeError: 'Sequential' object has no attribute 'predict_proba'

## BeeSwarm Plots

In [ ]:
for name, sv in shap_values.items():
    shap.plots.beeswarm(sv, max_display=15, show=False)
    plt.title(name + " - SHAP beeswarm")
    plt.show()

## Violin Plots

In [ ]:
for name, sv in shap_values.items():
    shap.summary_plot(sv.values, X_eval, plot_type="violin", max_display=15, show=False)
    plt.title(name + " - SHAP violin")
    plt.show()

## Comparisons

In [ ]:
importance = pd.DataFrame(
    {name: np.abs(sv.values).mean(axis=0) for name, sv in shap_values.items()},
    index=X_eval.columns,
)

importance = importance / importance.sum()

In [ ]:
top = importance.mean(axis=1).sort_values(ascending=False).head(15).index
top_imp = importance.loc[top]

In [ ]:
top_imp.iloc[::-1].plot(kind="barh", figsize=(10, 9), width=0.85)
plt.title("Top 15 features across models")
plt.xlabel("normalised mean |SHAP|")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
sns.heatmap(top_imp, annot=True, fmt=".3f", cmap="YlOrRd")
plt.title("Feature importance heatmap")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(importance.corr(method="spearman"), annot=True, fmt=".2f", cmap="Blues")
plt.title("Rank agreement between models")
plt.tight_layout()
plt.show()